# Calendar Table

Creates and populates `caresync_dbws.silver.calender` with standard date attributes for every day from 1940-01-01 through 2030-12-31.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_dbws.silver;

CREATE OR REPLACE TABLE caresync_dbws.silver.calender
USING DELTA
AS
WITH date_series AS (
  SELECT explode(sequence(to_date('1940-01-01'), to_date('2030-12-31'), interval 1 day)) AS calendar_date
)
SELECT
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS datekey,
  calendar_date AS `date`,
  year(calendar_date) AS `year`,
  quarter(calendar_date) AS quarter,
  concat('Q', quarter(calendar_date)) AS quarter_name,
  month(calendar_date) AS month,
  date_format(calendar_date, 'MMMM') AS month_name,
  date_format(calendar_date, 'MMM') AS month_short_name,
  CAST(date_format(calendar_date, 'yyyyMM') AS INT) AS year_month_key,
  date_format(calendar_date, 'yyyy-MM') AS year_month,
  concat(year(calendar_date), '-Q', quarter(calendar_date)) AS year_quarter,
  day(calendar_date) AS day,
  dayofweek(calendar_date) AS day_of_week,
  weekday(calendar_date) + 1 AS iso_day_of_week,
  date_format(calendar_date, 'EEEE') AS day_name,
  date_format(calendar_date, 'EEE') AS day_short_name,
  dayofyear(calendar_date) AS day_of_year,
  weekofyear(calendar_date) AS week,
  date_sub(calendar_date, weekday(calendar_date)) AS week_start_date,
  date_add(date_sub(calendar_date, weekday(calendar_date)), 6) AS week_end_date,
  CAST(date_trunc('month', calendar_date) AS DATE) AS month_start_date,
  last_day(calendar_date) AS month_end_date,
  CAST(day(last_day(calendar_date)) AS INT) AS days_in_month,
  CAST(date_trunc('quarter', calendar_date) AS DATE) AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('quarter', calendar_date) AS DATE), 3), 1) AS quarter_end_date,
  make_date(year(calendar_date), 1, 1) AS year_start_date,
  make_date(year(calendar_date), 12, 31) AS year_end_date,
  CASE WHEN weekday(calendar_date) < 5 THEN true ELSE false END AS is_weekday,
  CASE WHEN weekday(calendar_date) >= 5 THEN true ELSE false END AS is_weekend,
  CASE WHEN day(calendar_date) = 1 THEN true ELSE false END AS is_month_start,
  CASE WHEN calendar_date = last_day(calendar_date) THEN true ELSE false END AS is_month_end,
  CASE WHEN calendar_date = CAST(date_trunc('quarter', calendar_date) AS DATE) THEN true ELSE false END AS is_quarter_start,
  CASE WHEN calendar_date = date_sub(add_months(CAST(date_trunc('quarter', calendar_date) AS DATE), 3), 1) THEN true ELSE false END AS is_quarter_end,
  CASE WHEN calendar_date = make_date(year(calendar_date), 1, 1) THEN true ELSE false END AS is_year_start,
  CASE WHEN calendar_date = make_date(year(calendar_date), 12, 31) THEN true ELSE false END AS is_year_end,
  CASE
    WHEN day(calendar_date) IN (1, 21, 31) THEN concat(day(calendar_date), 'st')
    WHEN day(calendar_date) IN (2, 22) THEN concat(day(calendar_date), 'nd')
    WHEN day(calendar_date) IN (3, 23) THEN concat(day(calendar_date), 'rd')
    ELSE concat(day(calendar_date), 'th')
  END AS day_suffix
FROM date_series;


In [0]:
%sql
SELECT
  COUNT(*) AS row_count,
  MIN(`date`) AS min_date,
  MAX(`date`) AS max_date,
  DATEDIFF(MAX(`date`), MIN(`date`)) + 1 AS expected_row_count,
  COUNT(*) - (DATEDIFF(MAX(`date`), MIN(`date`)) + 1) AS row_count_delta,
  MIN(datekey) AS min_datekey,
  MAX(datekey) AS max_datekey,
  SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) AS weekend_days,
  SUM(CASE WHEN is_weekday THEN 1 ELSE 0 END) AS weekday_days
FROM caresync_dbws.silver.calender;

WITH ordered_dates AS (
  SELECT
    `date`,
    LEAD(`date`) OVER (ORDER BY `date`) AS next_date
  FROM caresync_dbws.silver.calender
)
SELECT COUNT(*) AS missing_day_gaps
FROM ordered_dates
WHERE next_date IS NOT NULL
  AND DATEDIFF(next_date, `date`) <> 1;

SELECT
  datekey,
  `date`,
  day_name,
  month_name,
  week,
  quarter,
  `year`,
  is_month_start,
  is_month_end,
  is_year_start,
  is_year_end
FROM caresync_dbws.silver.calender
WHERE `date` IN (DATE '1940-01-01', DATE '2000-02-29', DATE '2024-02-29', DATE '2030-12-31')
ORDER BY `date`;
